# Exploração de Dados — Dirty Cafe Sales

Este notebook realiza a **exploração inicial** de uma tabela de vendas de cafeteria (`workspace.my_database.dirty_cafe_sales_table`) com aproximadamente **10 mil linhas**. O objetivo é identificar inconsistências, valores inválidos e dados faltantes antes da etapa de limpeza.

## Etapas do notebook

1. **Abertura e informações gerais** — Carrega a tabela no Spark, conta as linhas, exibe as primeiras 5 linhas, o schema e os nomes das colunas.

2. **Quantidade de valores únicos por coluna** — Conta os valores distintos de cada coluna. Revela que `transaction_id` não tem duplicatas (10 mil únicos para 10 mil linhas) e que as demais colunas têm poucos valores únicos, facilitando a identificação de inconsistências.

3. **Visualização dos valores únicos por coluna** — Lista todos os valores distintos de cada coluna (exceto `transaction_id`). Conclui que os valores inválidos são: `ERROR`, `UNKNOWN` e `NULL`.

4. **Exploração dos IDs** — Verifica se os IDs seguem um padrão coerente separando `transaction_id` em prefixo e sufixo pelo delimitador `_`. Conclui que todos os IDs seguem o padrão `TXN_<id único>` (prefixo sempre `TXN` e 10 mil sufixos distintos), sem duplicatas.

5. **Identificação de todas as linhas com valores inválidos** — Conta os valores inválidos (`ERROR`, `UNKNOWN` ou `NULL`) por coluna e exibe todas as linhas afetadas. Observa que:
   - Todas as colunas, exceto `transaction_id`, contêm dados inválidos/faltantes.
   - A proporção varia de ~5% a ~40% dependendo da coluna.
   - Algumas linhas possuem mais de uma coluna inválida.

> O tratamento dessas e outras inconsistências é feito no notebook **`2_data_cleaning`**.


### Abertura e exibição de informações gerais

In [0]:
data_path = 'workspace.my_database.dirty_cafe_sales_table'

df = spark.read.table(data_path)

df_len = df.count() # conta as linhas
print(f'DATA LENGHT: {df_len}')

df.show(5)
df.printSchema()

print(df.columns)

### Exploração da quantidade de valores únicos por coluna

In [0]:
print("Total data :", df_len)

for col in df.columns:
    # o select bascimente é um select do sql, que sleeciona a coluna passada
    # o distinct é equivalente à unique
    print(col, ':', df.select(col).distinct().count())

# Esses valores já indicam que não há duplicatas no transaction_id, pois são 10 mil valores distintos, e o total de dados é 10 mil

# Também é possível veirificar que, à exceção do de transaction_date (e o próprio transaction_id), todos os outros possuem poucos valores únicos, facilictando a identificação de valores errôneos ou vazios.


### Visualização dos valores únicos por coluna (exceto o ID)

In [0]:
for col in df.columns:
    if col != 'transaction_id':
        unique = df.select(col).distinct()
        unique_counts = unique.count()
        unique.show(unique_counts) # para exibir o número exato de linhas, pois o default é 20

# Apesar da verificação anterior que o transaction_date possui uma quantidade elevada de valores únicos, não é uma quantidade tão grande a ponto de ficar inviável a exibição

# Através da verificação dos valores, é possível verificar que os inválidos são: ERROR, UNKNOWN, e NULL

### Exploração dos IDs

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, split, regexp_extract, countDistinct

def verify_ids(df):
    '''
    Função que verifica se os IDs seguem o padrão: LETRA(NÚMERO)_ restante
    Ex.: 'TXN_0001', 'A001_abc', etc. — prefixo de letras seguido de números, separados por '_' do restante
    '''

    # --- 1. Separar em prefixo e sufixo usando '_' como delimitador ---
    df_ids = df.select(
        'transaction_id',
        split(col('transaction_id'), '_').getItem(0).alias('prefix'),
        split(col('transaction_id'), '_').getItem(1).alias('suffix')
    )

    print('Amostra dos IDs separados em prefixo e sufixo:')
    df_ids.show(10, truncate=False)

    # 2. Conta os valores únicos pós a separação entre prefixo e sufixo
    print('Valores únicos pós a separação entre prefixo e sufixo:')
    unique_prefix = df_ids.select(col('prefix')).distinct()
    unique_counts_prefix = unique_prefix.count()
    print('N unique prefixes:', unique_counts_prefix)

    unique_suffix = df_ids.select(col('suffix')).distinct()
    unique_counts_suffix = unique_suffix.count()
    print('N unique suffixes:', unique_counts_suffix)

In [0]:
# Foi verificado que os Ids seguem o padrão "TXN + _ + id único", pois o único prefixo encontrado foi 'TXN', e todos os sufixos são únicos (10 ml valores)

verify_ids(df)

### Identificação de todas as linhas com valores inválidos

In [0]:
# Primeiro é realizada a contagem de inválidos ('ERROR', 'UNKNOWN', ou None) por coluna. Depois é aplicado um filtro que busca todas as linhas com inválidos, e exibe em forma de tabela todas aquelas identificadas

# isin() não captura NULLs no PySpark, por isso .isNull() é adicionado separadamente
string_invalids = ['ERROR', 'UNKNOWN']

print(f'Number of invalid values by column') 
for col in df.columns:
    n_invalids = df.filter(df[col].isin(string_invalids) | df[col].isNull()).count()
    print(col, ':', n_invalids)

display(
    df.filter(
    (df['transaction_id'].isin(string_invalids) | df['transaction_id'].isNull()) |
    (df['item'].isin(string_invalids) | df['item'].isNull()) |
    (df['quantity'].isin(string_invalids) | df['quantity'].isNull()) |
    (df['price_per_unit'].isin(string_invalids) | df['price_per_unit'].isNull()) |
    (df['total_spent'].isin(string_invalids) | df['total_spent'].isNull()) |
    (df['payment_method'].isin(string_invalids) | df['payment_method'].isNull()) |
    (df['location'].isin(string_invalids) | df['location'].isNull()) |
    (df['transaction_date'].isin(string_invalids) | df['transaction_date'].isNull())
    )
)

# Observa-se que, à exceção do id, todos possuem dados inválidos/faltantes, com uma variação considerável de quantidade dependendo da coluna. As com menos dados inválidos ficam em torno de 5%, as com mais, variam entre 30 e 40%

# Além disso, observa-se pela tabela, que algumas linhas possuem mais de uma coluna inválida

# O notebook 2_data_cleaning objetiva tratar essas e outras inconsistências